# Three-agent comparison on the short route

Compare Expected SARSA, DQN, and Dyna-DQN on the 300 m / 16 s route using 60 held-out trials per policy. Run All reads the bundled results. Set `RUN_TRAINING = True` for a fresh run.

In [ ]:
from pathlib import Path
import os
import sys

os.environ.setdefault("SDL_AUDIODRIVER", "dummy")
os.environ.setdefault("PYGAME_HIDE_SUPPORT_PROMPT", "1")
here = Path.cwd().resolve()
repo = next((p for p in (here, *here.parents) if (p / "runs/index.csv").is_file()), None)
if repo is None:
    raise RuntimeError("Open this notebook from the MORL4AVSO checkout.")
sys.path.insert(0, str(repo))


import json

import pandas as pd
import torch
from IPython.display import display

from algorithms.dqn_agent import DQNAgent
from algorithms.sensor_agent import LinearExpectedSarsa
from environments.sensor_highway import ExperimentConfig
from experiments.route_notebook_helpers import evaluate_route, new_run_directory, summarize_trials
from experiments.run_long_dqn import train_policy

In [ ]:
RUN_TRAINING = False
TRAIN_EPISODES = None  # None uses the recorded episode budget.
POLICIES = ('sarsa', 'dqn', 'dyna_dqn')

bundled = repo / "runs/comparisons/dqn_long_training_20261112_170403_004536"
saved_run = os.environ.get("FINAL_PROJECT_DQN_RUN_DIR")
recorded = Path(saved_run).expanduser().resolve() if saved_run else bundled
config = ExperimentConfig.load(recorded / "config.json")
if config != ExperimentConfig.load(bundled / "config.json"):
    raise ValueError("The saved run uses a different environment configuration.")
saved_settings = json.loads((recorded / "run_settings.json").read_text())
protocol = {key: saved_settings[key] for key in (
    "training_seed", "test_seeds", "test_weather", "dqn_settings", "sarsa_settings",
)}
protocol.update(policies=list(POLICIES), status="training", torch_threads=1)
if TRAIN_EPISODES is not None:
    protocol["dqn_settings"]["episodes"] = TRAIN_EPISODES
    protocol["sarsa_settings"]["episodes"] = TRAIN_EPISODES

## Matched evaluation

Finish time averages successful trips. Sensing power averages all trips.

In [ ]:
if RUN_TRAINING:
    torch.set_num_threads(1)
    output = new_run_directory(repo, "dqn_short_route")
    config.save(output / "config.json")
    (output / "run_settings.json").write_text(json.dumps(protocol, indent=2) + "\n")
    for policy in POLICIES:
        train_policy(policy, output)
    weights = output
else:
    weights = recorded

if RUN_TRAINING or not (recorded / "evaluation.csv").is_file():
    if not RUN_TRAINING:
        output = new_run_directory(repo, "frozen_route_evaluation")
        config.save(output / "config.json")
        settings = dict(protocol, source_run=str(recorded))
        (output / "run_settings.json").write_text(json.dumps(settings, indent=2) + "\n")
    agents = {
        "Expected SARSA": LinearExpectedSarsa.load(weights / "sarsa.npz"),
        "DQN": DQNAgent.load(weights / "dqn.pt"),
        "Dyna-DQN": DQNAgent.load(weights / "dyna_dqn.pt"),
    }
    agents.update({"idle_all_on": None, "random_all_on": None})
    trials = evaluate_route(config, agents, protocol)
    trials.to_csv(output / "evaluation.csv", index=False)
    summarize_trials(trials).to_csv(output / "summary.csv")
    if RUN_TRAINING:
        protocol["status"] = "complete"
        (output / "run_settings.json").write_text(json.dumps(protocol, indent=2) + "\n")
    print(f"Saved to {output.relative_to(repo)}")
else:
    trials = pd.read_csv(recorded / "evaluation.csv")

display(summarize_trials(trials).round(3))